<div style="background: linear-gradient(135deg, #6610f2 0%, #0d6efd 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">🎯 Mini Projeto — Pipeline CSV (Bloco 01)</h1>
  <p style="color: #d3c7ff; margin: 8px 0 0 0; font-size: 1.1em;">Sem pandas · Apenas Python puro · Fase 00 · Bloco 01</p>
</div>

## 🎯 Objetivo

Construir um pipeline completo de processamento de CSV **sem usar pandas**:

1. Gerar dataset sintético
2. Ler o CSV linha a linha
3. Limpar dados inválidos
4. Filtrar por condição
5. Calcular estatísticas
6. Salvar resultado

---

## 📋 Contexto

Você tem um arquivo de log de treino de modelos com: epoch, loss, val_loss, accuracy. Precisa:
- Remover linhas corrompidas (valores ausentes ou inválidos)
- Encontrar a melhor epoch (menor val_loss)
- Salvar apenas as epochs onde val_loss < threshold

---

In [ ]:
import os, csv, random
random.seed(42)

# ═══ ETAPA 1: Gerar dataset sintético ════════════════════════════════════
def gerar_log_treino(caminho: str, n_epochs: int = 50, taxa_corrupcao: float = 0.1):
    """
    Gera um CSV simulando log de treino com algumas linhas corrompidas.

    Args:
        caminho: Onde salvar o CSV
        n_epochs: Número de épocas
        taxa_corrupcao: Proporção de linhas com dados inválidos
    """
    os.makedirs(os.path.dirname(caminho), exist_ok=True)

    with open(caminho, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "loss", "val_loss", "accuracy"])

        loss     = 1.5
        val_loss = 1.6
        acc      = 0.3

        for ep in range(1, n_epochs + 1):
            # Simular convergência
            loss     *= random.uniform(0.88, 0.97)
            val_loss *= random.uniform(0.87, 0.98)
            acc      = min(0.99, acc + random.uniform(0.005, 0.025))

            # Introduzir corrupção aleatória
            if random.random() < taxa_corrupcao:
                opcoes_corrompidas = [
                    [ep, "", val_loss, acc],           # loss vazia
                    [ep, loss, "NaN", acc],             # val_loss inválido
                    [ep, -loss, val_loss, acc],          # loss negativa
                    [ep, loss, val_loss, ""]             # acc vazia
                ]
                linha = random.choice(opcoes_corrompidas)
            else:
                linha = [ep, f"{loss:.6f}", f"{val_loss:.6f}", f"{acc:.4f}"]

            writer.writerow(linha)

    print(f"✅ Dataset gerado: {caminho} ({n_epochs} linhas)")

gerar_log_treino("data/treino_log.csv", n_epochs=50)

In [ ]:
# ═══ ETAPA 2: Ler CSV ════════════════════════════════════════════════════
def ler_log(caminho: str) -> tuple:
    """
    Lê o CSV de log e retorna (cabecalho, linhas_brutas).

    Returns:
        (cabecalho: list, dados: list[dict])
    """
    dados = []
    with open(caminho, 'r', encoding='utf-8') as f:
        leitor = csv.DictReader(f)
        for linha in leitor:
            dados.append(dict(linha))
    return dados

dados_brutos = ler_log("data/treino_log.csv")
print(f"Linhas lidas: {len(dados_brutos)}")
print(f"Primeiras 3:")
for linha in dados_brutos[:3]:
    print(f"  {linha}")

In [ ]:
# ═══ ETAPA 3: Limpar dados ════════════════════════════════════════════════
def validar_linha(linha: dict) -> bool:
    """
    Retorna True se a linha é válida.
    Critérios:
    - Nenhum campo vazio
    - loss e val_loss devem ser float positivos
    - accuracy entre 0 e 1
    """
    try:
        epoch    = int(linha["epoch"])
        loss     = float(linha["loss"])
        val_loss = float(linha["val_loss"])
        acc      = float(linha["accuracy"])

        if loss <= 0 or val_loss <= 0:
            return False
        if not (0.0 <= acc <= 1.0):
            return False
        return True
    except (ValueError, KeyError):
        return False

def limpar_dados(dados: list) -> tuple:
    """
    Remove linhas inválidas e converte tipos.

    Returns:
        (dados_limpos: list[dict], n_removidos: int)
    """
    limpos    = []
    removidos = 0

    for linha in dados:
        if validar_linha(linha):
            limpos.append({
                "epoch":    int(linha["epoch"]),
                "loss":     float(linha["loss"]),
                "val_loss": float(linha["val_loss"]),
                "accuracy": float(linha["accuracy"])
            })
        else:
            removidos += 1

    return limpos, removidos

dados_limpos, n_removidos = limpar_dados(dados_brutos)
print(f"Linhas originais:  {len(dados_brutos)}")
print(f"Linhas válidas:    {len(dados_limpos)}")
print(f"Linhas removidas:  {n_removidos}")

In [ ]:
# ═══ ETAPA 4: Filtrar e Analisar ══════════════════════════════════════════
def encontrar_melhor_epoch(dados: list) -> dict:
    """Retorna a linha com menor val_loss."""
    return min(dados, key=lambda x: x["val_loss"])

def filtrar_por_val_loss(dados: list, threshold: float) -> list:
    """Retorna apenas as epochs com val_loss abaixo do threshold."""
    return [d for d in dados if d["val_loss"] < threshold]

def calcular_estatisticas(dados: list) -> dict:
    """Calcula estatísticas descritivas do log de treino."""
    losses    = [d["loss"] for d in dados]
    val_losses = [d["val_loss"] for d in dados]
    acuracias = [d["accuracy"] for d in dados]

    return {
        "n_epochs":         len(dados),
        "loss_final":       losses[-1],
        "val_loss_final":   val_losses[-1],
        "val_loss_minimo":  min(val_losses),
        "acuracia_maxima":  max(acuracias),
        "acuracia_final":   acuracias[-1]
    }

melhor = encontrar_melhor_epoch(dados_limpos)
stats  = calcular_estatisticas(dados_limpos)

print("\n══ Melhor Epoch ══════════════════════")
print(f"  Epoch:    {melhor['epoch']}")
print(f"  Val Loss: {melhor['val_loss']:.6f}")
print(f"  Accuracy: {melhor['accuracy']:.4f}")

print("\n══ Estatísticas Gerais ═══════════════")
for k, v in stats.items():
    print(f"  {k:25} {v:.6f}" if isinstance(v, float) else f"  {k:25} {v}")

In [ ]:
# ═══ ETAPA 5: Salvar resultados ══════════════════════════════════════════
def salvar_csv(dados: list, caminho: str, campos: list = None):
    """
    Salva lista de dicts como CSV.

    Args:
        dados: Lista de dicts com os dados
        caminho: Caminho de saída
        campos: Colunas a incluir (None = todas)
    """
    if not dados:
        print("⚠️ Nenhum dado para salvar")
        return

    campos = campos or list(dados[0].keys())

    with open(caminho, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=campos, extrasaction='ignore')
        writer.writeheader()
        for linha in dados:
            writer.writerow(linha)

    print(f"✅ Salvo: {caminho} ({len(dados)} linhas)")

# Salvar dados limpos
salvar_csv(dados_limpos, "data/treino_log_limpo.csv")

# Salvar apenas top epochs
threshold_val_loss = stats["val_loss_minimo"] * 1.05  # 5% acima do mínimo
top_epochs = filtrar_por_val_loss(dados_limpos, threshold_val_loss)
salvar_csv(top_epochs, "data/treino_top_epochs.csv")

print(f"\nTop epochs (val_loss < {threshold_val_loss:.4f}): {len(top_epochs)} encontradas")

In [ ]:
# ═══ ETAPA 6: Visualizar progresso (ASCII Plot) ═══════════════════════════
def ascii_plot(valores: list, titulo: str, largura: int = 40):
    """Plota uma lista de valores como gráfico ASCII."""
    minv = min(valores)
    maxv = max(valores)
    intervalo = maxv - minv or 1

    print(f"\n{'═' * (largura + 12)}")
    print(f"  {titulo}")
    print(f"  Max: {maxv:.4f} | Min: {minv:.4f}")
    print(f"{'═' * (largura + 12)}")

    for i, val in enumerate(valores):
        frac = (val - minv) / intervalo
        barra = int(frac * largura)
        print(f"  E{i+1:3d} | {'█' * barra:{largura}} | {val:.4f}")

# Mostrar loss e accuracy
losses    = [d["loss"] for d in dados_limpos]
val_losses = [d["val_loss"] for d in dados_limpos]
acuracias = [d["accuracy"] for d in dados_limpos]

ascii_plot(losses[:20], "Training Loss (primeiras 20 epochs)", largura=30)
ascii_plot(acuracias[:20], "Accuracy (primeiras 20 epochs)", largura=30)

---

## ✅ Critério de Conclusão do Bloco 01

**Este mini projeto está completo quando você:**

- [ ] Executou todas as células sem erro
- [ ] Entende cada função sem consultar nada
- [ ] Consegue modificar a lógica de validação (`validar_linha`) sem ajuda
- [ ] Consegue adicionar uma nova coluna estatística em `calcular_estatisticas`

**🚀 Desafio Extra:** Adicione uma função que detecta `overfitting` — quando `val_loss` começa a subir enquanto `loss` continua caindo. Retorne a epoch onde o overfitting começou.

---

➡️ **Próximo bloco:** `../Bloco_02_Python_Intermediario/01_funcoes_avancadas.ipynb`